In [2]:
import osmnx as ox
import networkx as nx
import pandas as pd
import numpy as np
import math
import torch
import random
from shapely import wkt
from shapely.geometry import LineString, Point
import ast
# grafik
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# limit artırma
import sys
sys.setrecursionlimit(2000)
# Limiti 2000'e çıkarır
# özyineleme hata ayıklama içi
import pysnooper

In [3]:
data=pd.read_csv('kocaeli_ana_yollar_final_v15.csv')
print(data.head())

   trip_id      trip_name  segment_id         osm_segment_id  length_meters  \
0        1  Izmit - Gebze        1001  5661714310-9449574131      23.951446   
1        1  Izmit - Gebze        1002  9449574131-6520959031      11.933984   
2        1  Izmit - Gebze        1003  6520959031-2363062636      17.363631   
3        1  Izmit - Gebze        1004  2363062636-1295374326      24.627443   
4        1  Izmit - Gebze        1005   1295374326-968511694     177.067344   

      road_type  lanes lanes_kaynagi  tunnel  max_speed   hiz_kaynagi  \
0       primary      2    varsayilan       0         50           osm   
1  primary_link      2           osm       0         50    varsayilan   
2       primary      2           osm       0         75  yumusatilmis   
3       primary      2    varsayilan       0         90    varsayilan   
4       primary      3           osm       0         90    varsayilan   

   has_traffic_light                                             coords  \
0          

In [4]:
def anomaly_definition(anomaly_type,anomaly_time,fnl_tm):
    anomaly_typs=[1,2,3,4,5,6,7,8,9]
    anomaly_stts=[0,1]
    stts_wghs=[92,8]
    typs_wghs=[30,10,25,1,14,4,4,4,8]
    """
    1- hız sınırı aşımı
    2-olması gerekenden fazla bekleme?
    3-yüksek ivme
    4-kaza
    5-yanlış açı
    6-gps(hız)
    7-gps(ivme)
    8-gps(açı)
    9-gps(donma)
    """
    
    if fnl_tm<=0:
        anomaly_type=0
    if anomaly_type==0:
        anomaly_prb=random.choices(anomaly_stts, weights=stts_wghs, k=1)[0]
        anomaly_type=anomaly_prb
        anomaly_time=0
    else:
        return anomaly_type,anomaly_time,fnl_tm
    if anomaly_prb==1:
        anomaly_type=random.choices(anomaly_typs, weights=typs_wghs, k=1)[0]
        match anomaly_type:
            case 1:
                anomaly_time=np.random.lognormal(mean=np.log(10),sigma=.3)
                anomaly_time=np.clip(anomaly_time,1,17)
                anomaly_time=int(anomaly_time)
                fnl_tm=anomaly_time
                return anomaly_type,anomaly_time,fnl_tm
            case 2:
                anomaly_time=np.random.lognormal(mean=np.log(150),sigma=.4)
                anomaly_time=np.clip(anomaly_time,60,600)
                anomaly_time=int(anomaly_time)
                fnl_tm=anomaly_time
                return anomaly_type,anomaly_time,fnl_tm
            case 3:
                anomaly_time=np.random.normal(loc=2,scale=1)
                anomaly_time=np.clip(anomaly_time,1,5)
                anomaly_time=int(anomaly_time)
                fnl_tm=anomaly_time
                return anomaly_type,anomaly_time,fnl_tm
            case 4:
                accdnt_svrty = np.random.choice(['will_cnteu', 'will_rec', 'cant_cnteu'], p=[0.45, 0.50, 0.05])
                if accdnt_svrty=='will_cnteu':
                    anomaly_time=np.random.normal(loc=150,scale=40)
                if accdnt_svrty=='will_rec':
                    anomaly_time=np.random.lognormal(mean=np.log(750),sigma=.2)
                    anomaly_time=np.clip(anomaly_time,550,1450)
                if accdnt_svrty=='cant_cnteu':
                    anomaly_time=np.random.normal(loc=3000,scale=500)
                    anomaly_time = np.clip(anomaly_time, 1500, 5000)
                anomaly_time=int(anomaly_time)
                fnl_tm=anomaly_time
                return anomaly_type,anomaly_time,fnl_tm
            case 5 | 6 | 7 | 8: 
                anomaly_time = np.random.randint(1,5)
                fnl_tm=anomaly_time
                return anomaly_type,anomaly_time,fnl_tm
            case 9:
                anomaly_time=np.random.randint(5,16)
                fnl_tm=anomaly_time
                return anomaly_type,anomaly_time,fnl_tm
    else:
        return anomaly_type,anomaly_time,fnl_tm


In [5]:
def distance_calc(avrgspd):
    return (avrgspd/3.6) 
def bearing_calc(dist,lng,geometry,bearing,anomaly_type,anomaly_time,fnl_tm):
    geometry.interpolate()
    geometry_shp=LineString(geometry)
    if lng<0:
        return bearing
    if lng<dist:
        dist = lng - 0.001
        rt=dist/lng
    else:
        rt=dist/lng
    if anomaly_type==4 and anomaly_time>1450 and anomaly_time-5>=fnl_tm:
         return bearing
    rt_crd= geometry_shp.interpolate(rt, normalized=True)
    rt_crd_bfr=geometry_shp.interpolate(rt+0.001, normalized=True)
    x_1, y_1=rt_crd.x , rt_crd.y
    x_2, y_2=rt_crd_bfr.x , rt_crd_bfr.y
    d_x =x_2 - x_1
    d_y = y_2 - y_1
    rdn = math.atan2(d_x, d_y)
    degrs= math.degrees(rdn)
    bearing=(degrs + 360) % 360
    d=np.random.lognormal(mean=np.log(1.4),sigma=.35)
    opr=np.random.choice([-1, 1])
    bearing=bearing+d*opr
    if anomaly_type==4 and anomaly_time-3<fnl_tm:
        wrong_brng=np.random.normal(loc=24,scale=8)
        opr=np.random.choice([-1, 1])
        bearing=bearing+wrong_brng*opr
    if anomaly_type==4 and anomaly_time-8>=fnl_tm:
        wrong_brng=5
        opr=np.random.choice([-1, 1])
        bearing=bearing+wrong_brng*opr
    if anomaly_type==4 and anomaly_time>1450 and anomaly_time-5<fnl_tm:
        bearing=np.random.normal(loc=180,scale=60)
    if anomaly_type==5:
        wrong_brng=np.random.normal(loc=25,scale=5)
        opr=np.random.choice([-1, 1])
        bearing=bearing+wrong_brng*opr
    if bearing<0:
        return bearing_calc(dist,lng,geometry,bearing,anomaly_type,anomaly_time,fnl_tm)
    return bearing 
def sentetic_statue(statue,traffic):
    traffic=traffic*10
    stts=[1,2,3]
    I1=[[95,2,3],
      [25,70,5],
      [20,4,71]]
    I2=[[20,10,70,0],
      [10,40,50,0],
      [5,5,40,50],
        [0,20,0,80]]
    a,b=0,0 
    t=traffic
    lnr=a*t-b
    I3=[[0,0,0],[0,0,0],[0,0,0]]
    for x in range(0,3):
        for y in range(0,3):
            a=(I2[x][y]-I1[x][y])/(70-30)
            b=I1[x][y]-a*30
            I3[x][y]=a*t+b
    if traffic<=30: 
        if statue==1:
            wghs=I1[0]
            statue_new=random.choices(stts, weights=wghs, k=1)[0]
            return statue_new
        if statue==2:
            wghs=I1[1]
            statue_new=random.choices(stts, weights=wghs, k=1)[0]
            return statue_new
        if statue==3:
            wghs=I1[2]
            statue_new=random.choices(stts, weights=wghs, k=1)[0]
            return statue_new
    if 30<traffic<70:
        if statue==1:
            wghs=I3[0]
            statue_new=random.choices(stts, weights=wghs, k=1)[0]
            return statue_new
        if statue==2:
            wghs=I3[1]
            statue_new=random.choices(stts, weights=wghs, k=1)[0]
            return statue_new
        if statue==3:
            wghs=I3[2]
            statue_new=random.choices(stts, weights=wghs, k=1)[0]
            return statue_new
    if traffic>=70:
        stts=[1,2,3,4]
        if statue==1:
            wghs=I2[0]
            statue_new=random.choices(stts, weights=wghs, k=1)[0]
            return statue_new
        if statue==2:
            wghs=I2[1]
            statue_new=random.choices(stts, weights=wghs, k=1)[0]
            return statue_new
        if statue==3:
            wghs=I2[2]
            statue_new=random.choices(stts, weights=wghs, k=1)[0]
            return statue_new
        if statue==4:
            wghs=I2[3]
            statue_new=random.choices(stts, weights=wghs, k=1)[0]
            return statue_new
 
def accel_calc(pr_spd,speed):
    return (speed-pr_spd)/3.6
     
def sentetic_tarffic_ratio(traffic):
    mean=np.random.normal(loc=2.9, scale=.5)
    traffic_new= np.random.lognormal(mean=np.log(mean),sigma=1.1)
    traffic_new= np.clip(traffic_new, 1, 9)
    if -3.5> traffic-traffic_new or traffic-traffic_new >3.5:
        return sentetic_tarffic_ratio(traffic)
    return traffic_new
def sentetic_speed(maxspeed,speed,statue,traffic,anomaly_type,able_ovrspd):
    if anomaly_type==1 and traffic>=7:
        anomaly_type=0
    avgspeed=(maxspeed-maxspeed*.08)*(1-(traffic/10)**4)
    if anomaly_type==2 and speed>=10:
        statue=3
    if anomaly_type==2 and speed<10:
        statue=4
    if statue==4:
        d=np.random.normal(loc=0, scale=0.33)
        d=np.clip(d,.01,.89)
        speed_new=0+d
        accl=accel_calc(speed,speed_new)
        
        if 1.5<accl or accl<-3:
            statue=3
            return sentetic_speed(maxspeed,speed,statue,traffic,anomaly_type,able_ovrspd)
        return speed_new,accl,statue,anomaly_type
    if speed/avgspeed<.6:
            statue=2
    if avgspeed-speed<-25:
            statue=3
    if speed>(maxspeed*1.1) and able_ovrspd==1:
        statue=3
    if anomaly_type==1 and speed>(maxspeed*1.1+5):
        statue=3
    if anomaly_type==1 and speed<(maxspeed*1.1):
        statue=2
    if statue==1:
        sigma=.5*speed/100
        d=np.random.normal(loc=sigma/2, scale=sigma)
        speed_new=speed+d
        accl=accel_calc(speed,speed_new)
        if speed_new<0:
            return sentetic_speed(maxspeed,speed,statue,traffic,anomaly_type,able_ovrspd)
        while speed_new>(maxspeed*1.1):
            print("hız: speed",speed,"yeni hız: ",speed_new,"durum:",statue)
            if anomaly_type==1 and speed_new<(maxspeed*1.1+8):
                return speed_new,accl,statue,anomaly_type
            statue=3
            return sentetic_speed(maxspeed,speed,statue,traffic,anomaly_type,able_ovrspd)
        return speed_new,accl,statue,anomaly_type

    if statue==2:
        if anomaly_type==3:
            d=np.random.normal(loc=7.2,scale=.4)
            speed_new=speed+d
        else: 
            d=np.random.normal(loc=3,scale=.8)
            d=np.clip(d,1.5,5.4)
            speed_new=speed+d
        
        if speed>=speed_new:
            return sentetic_speed(maxspeed,speed,statue,traffic,anomaly_type,able_ovrspd)
    if statue==3:
        if anomaly_type==3:
            d=np.random.normal(loc=16.2,scale=1.8)
            d=np.clip(d,10.8,21.6)
            speed_new=speed-d
        else:
            d=np.random.normal(loc=5.75,scale=1.5)
            speed_new=speed-d
        if speed<=speed_new:
            return sentetic_speed(maxspeed,speed,statue,traffic,anomaly_type,able_ovrspd)
    accl=accel_calc(speed,speed_new)
    if anomaly_type==3:
         while 2.5<accl or accl<-6 or -3<accl<1.5:
             
             return sentetic_speed(maxspeed,speed,statue,traffic,anomaly_type,able_ovrspd)
         return speed_new,accl,statue,anomaly_type
    else: 
        while 1.5<accl or accl<-3:
            return sentetic_speed(maxspeed,speed,statue,traffic,anomaly_type,able_ovrspd)
    while speed_new>(maxspeed*1.1):
        print("hız: speed",speed,"yeni hız: ",speed_new,"durum:",statue,"hız sınırı",maxspeed,"aşım izni:",able_ovrspd)
        if able_ovrspd:
            return speed_new,accl,statue,anomaly_type
        if anomaly_type==1 and speed_new<(maxspeed*1.1+8):
            return speed_new,accl,statue,anomaly_type
        statue=3
        return sentetic_speed(maxspeed,speed,statue,traffic,anomaly_type,able_ovrspd)
        
    return speed_new,accl,statue,anomaly_type


def sentetic_speed_accident(speed,anomaly_time):
    if anomaly_time<=550:
        if speed<=10:
            speed_new=0.1
        else:
            speed_new=np.random.lognormal(mean=np.log(speed-speed*.35),sigma=0.02)
        accl=accel_calc(speed,speed_new)
        return speed_new,accl
        
    if 550<anomaly_time<=1450:
        if speed<=10:
            speed_new=0.1
        else:
            speed_new=np.random.lognormal(mean=np.log(speed-speed*.55),sigma=0.02)
        accl=accel_calc(speed,speed_new)
        return speed_new,accl
        
    if 1450<anomaly_time:
        if speed<10:
            speed_new=0.1
        speed_new=np.random.lognormal(mean=np.log(speed-speed*.9),sigma=0.02)
        speed_new=np.clip(speed_new,.1,150)
        accl=accel_calc(speed,speed_new)
        return speed_new,accl

def percent_calc(lng,fnl_lng):
    fnl_lng_p=fnl_lng    
    if fnl_lng<0:
        fnl_lng_p=1
    way_rat=100-fnl_lng_p/lng*100
    return way_rat